In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "Code"))
from pathlib import Path
import pandas as pd

def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'LIS' / 'code'
mail_root = Path('/Users/jedrek/Library/Mail/V10')

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))


## Parse LA country emails → country-level DataFrame

Scans all emails with **"LA country YYYY"** in the subject. Each email contains `data_national_YEAR_SUBGROUP = { … }` blocks.

Subgroups include:
- **`persons`** / **`households`** – whole country totals
- **`p_sex_*`** / **`h_sex_*`** – by sex (person / household head)
- **`p_age_*`** / **`h_age_*`** – by age bracket
- **`p_edu_*`** / **`h_edu_*`** – by education level
- **`p_popclass_*`** / **`h_popclass_*`** – by settlement size class
- **`h_hhsize_*`** – by household size

One row = one (year, subgroup) combination.


In [2]:
import ast
import re
import email
import email.policy
from html.parser import HTMLParser
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone, timedelta

LIS_SENDER = "postbox@lisdatacenter.org"

# ── .emlx parsing (macOS Mail) ──────────────────────────────────────────────

def parse_emlx(path: Path):
    """Parse a macOS .emlx file (first line = byte count, then RFC 2822 email)."""
    try:
        raw = path.read_bytes()
        nl = raw.index(b"\n")
        bc = int(raw[:nl].strip())
        return email.message_from_bytes(raw[nl + 1 : nl + 1 + bc], policy=email.policy.default)
    except Exception:
        return None


def _decode_payload(part) -> str:
    payload = part.get_payload(decode=True)
    if not payload:
        return ""
    charset = part.get_content_charset() or "utf-8"
    return payload.decode(charset, errors="replace")


class _HTMLToText(HTMLParser):
    BLOCK_TAGS = {"p","div","br","li","tr","h1","h2","h3","h4","h5","h6",
                  "blockquote","pre","hr","table","thead","tbody","tfoot"}
    SKIP_TAGS  = {"script","style","head"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self._parts = []; self._skip = 0
    def handle_starttag(self, tag, attrs):
        t = tag.lower()
        if t in self.SKIP_TAGS: self._skip += 1
        elif t in self.BLOCK_TAGS: self._parts.append("\n")
    def handle_endtag(self, tag):
        t = tag.lower()
        if t in self.SKIP_TAGS: self._skip = max(0, self._skip - 1)
        elif t in self.BLOCK_TAGS: self._parts.append("\n")
    def handle_data(self, data):
        if self._skip == 0: self._parts.append(data)
    def get_text(self) -> str:
        text = "".join(self._parts)
        lines = [l.rstrip() for l in text.splitlines()]
        cleaned, blank = [], 0
        for l in lines:
            if l == "": blank += 1; (cleaned.append("") if blank <= 1 else None)
            else: blank = 0; cleaned.append(l)
        return "\n".join(cleaned).strip()


def get_body(msg) -> str:
    plain, html_parts = [], []
    if msg.is_multipart():
        for part in msg.walk():
            if part.is_multipart(): continue
            ct = part.get_content_type()
            if ct == "text/plain": plain.append(_decode_payload(part))
            elif ct == "text/html": html_parts.append(_decode_payload(part))
    else:
        ct = msg.get_content_type()
        if ct == "text/plain": plain.append(_decode_payload(msg))
        elif ct == "text/html": html_parts.append(_decode_payload(msg))
    if plain: return "\n".join(plain).strip()
    if html_parts:
        p = _HTMLToText(); p.feed("\n".join(html_parts)); return p.get_text()
    raw = msg.get_payload(decode=True)
    return raw.decode("utf-8", errors="replace").strip() if raw else ""


# ── Dict-block parsers ──────────────────────────────────────────────────────

def _parse_value(raw: str):
    raw = raw.strip()
    try:
        val = ast.literal_eval(raw)
        if isinstance(val, list) and len(val) == 1:
            v = val[0]
            return None if (isinstance(v, str) and v in ("N/A", "NA")) else v
        return val
    except Exception:
        return None


def _parse_dict_body(body: str) -> dict:
    out = {}
    for m in re.finditer(r"'([^']+)'\s*:\s*(\[[^\]]*\])", body):
        out[m.group(1)] = _parse_value(m.group(2))
    return out


# ── Country-specific block regex ────────────────────────────────────────────
# Matches: data_national_1986_persons = { … \n}
#          data_national_2007_h_hhsize_4os = { … \n}

_NATIONAL_RE = re.compile(
    r"data_national_(\d{4})_(\w+)\s*=\s*\{(.+?)\n\}",
    re.DOTALL,
)


def extract_national_dicts(text: str) -> list[dict]:
    """Find all data_national_YEAR_SUBGROUP blocks and return rows."""
    rows = []
    for m in _NATIONAL_RE.finditer(text):
        year_str, subgroup, body = m.group(1), m.group(2), m.group(3)
        row = _parse_dict_body(body)
        row["year"]     = int(year_str)
        row["subgroup"] = subgroup
        rows.append(row)
    return rows


print("Parsers ready.")


Parsers ready.


In [3]:
# ── Scan emails ──────────────────────────────────────────────────────────────

all_country_rows = []
country_mail_index = []

for emlx_path in mail_root.rglob("*.emlx"):
    if emlx_path.name.endswith(".partial.emlx"):
        continue
    raw_bytes = emlx_path.read_bytes()
    if b"postbox@lisdatacenter.org" not in raw_bytes:
        continue

    msg = parse_emlx(emlx_path)
    if msg is None:
        continue

    subject = str(msg.get("Subject", ""))
    # Must match "LA country YYYY" (not "LA country" in something longer like "LA country_xyz")
    subj_match = re.search(r"LA country (\d{4})\b", subject)
    if subj_match is None:
        continue
    if subj_match == re.search(r"job 1453993 LA country 2009", subject):
        print(f"Skipping email with wrong subject: {subject} (file: {emlx_path})")
        continue
    # Wrong subject: job 1453993 LA country 2009

    body = get_body(msg)
    rows = extract_national_dicts(body)
    if not rows:
        continue

    date_str = str(msg.get("Date", ""))
    try:
        mail_dt = parsedate_to_datetime(date_str)
        if mail_dt.tzinfo is None:
            mail_dt = mail_dt.replace(tzinfo=timezone.utc)
    except Exception:
        mail_dt = None

    for row in rows:
        row["_mail_subject"] = subject
        row["_mail_date"]    = mail_dt
    
    all_country_rows.extend(rows)
    country_mail_index.append({
        "subject": subject,
        "date":    mail_dt,
        "n_rows":  len(rows),
        "file":    str(emlx_path),
    })

print(f"Parsed {len(all_country_rows)} subgroup-year rows from {len(country_mail_index)} emails.\n")
print("Emails used:")
for e in sorted(country_mail_index, key=lambda x: (x["date"] or datetime.min.replace(tzinfo=timezone.utc))):
    print(f"  [{e['date']}]  {e['subject']:<42}  → {e['n_rows']} rows")


Parsed 1239 subgroup-year rows from 24 emails.

Emails used:
  [2026-03-09 18:19:44+01:00]  job 1453964 LA country 1986                 → 29 rows
  [2026-03-09 18:21:13+01:00]  job 1453965 LA country 1992                 → 37 rows
  [2026-03-09 18:22:46+01:00]  job 1453966 LA country 1995                 → 39 rows
  [2026-03-09 18:24:16+01:00]  job 1453967 LA country 1999                 → 54 rows
  [2026-03-09 18:25:46+01:00]  job 1453969 LA country 2004                 → 54 rows
  [2026-03-09 18:27:15+01:00]  job 1453970 LA country 2005                 → 54 rows
  [2026-03-09 18:28:44+01:00]  job 1453971 LA country 2006                 → 54 rows
  [2026-03-09 18:30:19+01:00]  job 1453972 LA country 2007                 → 54 rows
  [2026-03-09 18:55:54+01:00]  job 1453987 LA country 2008                 → 54 rows
  [2026-03-09 19:19:09+01:00]  job 1453994 LA country 2009                 → 54 rows
  [2026-03-09 19:19:43+01:00]  job 1453995 LA country 2010                 → 54 rows
  [2

In [4]:
# ── Deduplicate & build final DataFrame ──────────────────────────────────────
df_raw = pd.DataFrame(all_country_rows)

if df_raw.empty:
    print("No data extracted.")
else:
    df_raw = df_raw.sort_values("_mail_date", na_position="first")
    df_country = (
        df_raw
        .drop_duplicates(subset=["year", "subgroup"], keep="last")
        .drop(columns=["_mail_subject", "_mail_date"])
        .sort_values(["year", "subgroup"])
        .reset_index(drop=True)
    )

    skip_cols = {"year", "subgroup"}
    for col in df_country.columns:
        if col in skip_cols:
            continue
        df_country[col] = pd.to_numeric(df_country[col], errors="coerce")

    print(f"Final DataFrame: {df_country.shape[0]} rows × {df_country.shape[1]} columns")
    print(f"Years covered  : {sorted(df_country['year'].unique())}")
    print(f"Subgroups      : {sorted(df_country['subgroup'].unique())}")
    display(df_country)


Final DataFrame: 1239 rows × 188 columns
Years covered  : [np.int64(1986), np.int64(1992), np.int64(1995), np.int64(1999), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Subgroups      : ['h_age_15_19', 'h_age_20_24', 'h_age_20_29', 'h_age_25_29', 'h_age_30_34', 'h_age_30_39', 'h_age_35_39', 'h_age_40_44', 'h_age_40_49', 'h_age_45_49', 'h_age_50_54', 'h_age_50_59', 'h_age_55_59', 'h_age_60_', 'h_age_60_64', 'h_age_65_69', 'h_age_70_', 'h_edu_gimnazjalne_podstawowe_i_nizsze', 'h_edu_policealne_srednie_zawodowe', 'h_edu_srednie', 'h_edu_srednie_ogolnoksztalcace', 'h_edu_wyzsze', 'h_edu_zasadnicze_zawodowe', 'h_hhsize_1os', 'h_hhsize_2os', 'h_hhsize_3_4os', 'h_hhsize_3os', 'h_hhsize_4os', 'h_popclass_100k_500k', 'h_popcl

,year,pitotalnet_N_total,pitotalnet_Nw_total,pitotalnet_mean,pitotalnet_median,pitotalnet_p10,pitotalnet_p25,pitotalnet_p75,pitotalnet_p90,pitotalnet_p99,...,hilab_pens_N_Top_10,hilab_pens_N_Top_1,hilab_pens_Nw_Bottom_50,hilab_pens_Nw_P50_90,hilab_pens_Nw_Top_10,hilab_pens_Nw_Top_1,hilab_pens_share_Bottom_50,hilab_pens_share_P50_90,hilab_pens_share_Top_10,hilab_pens_share_Top_1
0,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,1.0,2.412485e+04,1.973852e+04,5482.9211,1096.5842,0.2768,0.4961,0.2271,0.0536
1,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,125.0,13.0,6.842686e+05,5.482921e+05,137073.0286,14255.5950,0.2783,0.4931,0.2286,0.0394
2,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,283.0,29.0,1.550570e+06,1.241333e+06,310333.3367,31800.9426,0.3098,0.4748,0.2155,0.0425
3,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,203.0,21.0,1.113033e+06,8.904264e+05,222606.5984,23028.2688,0.3147,0.4804,0.2049,0.0369
4,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,185.0,19.0,1.011051e+06,8.092792e+05,202868.0823,20835.1003,0.2635,0.4974,0.2391,0.0469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1234,2023,6870.0,4.116577e+06,46962.3173,42000.0,0.0,20400.0000,60000.0,90321.2422,216000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1235,2023,24918.0,1.246730e+07,28231.8837,27600.0,0.0,0.0000,42000.0,56400.0000,118800.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1236,2023,30644.0,1.638333e+07,29332.5921,30000.0,0.0,2590.0801,42000.0,55440.0000,114000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237,2023,26748.0,1.494165e+07,39800.7004,38400.0,0.0,14400.0000,54000.0,75949.2031,168000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ── Save ─────────────────────────────────────────────────────────────────────
save_path = data_root.parent / "LIS_Country_full.csv"
df_country.to_csv(save_path, index=False)
print(f"Saved to {save_path}")


Saved to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/LIS/LIS_Country_full.csv
